# LAB 03 — ACO para Projeto de Topologia de Rede de Baixa Latência

Implementação do zero de **Ant Colony Optimization (ACO)** para encontrar uma árvore geradora de baixa latência conectando 10 switches.

A solução possui exatamente 9 arestas, não contém ciclos e deve manter todos os switches conectados.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)

N = 10
RHO = 0.2
N_FORMIGAS = 30
N_ITER = 100
ALPHA = 1.0
BETA = 3.0
Q = 100.0

# Matriz simétrica de latências físicas (ms), com diagonal infinita.
D = np.array([
    [0, 12, 25, 18, 30, 22, 35, 28, 40, 32],
    [12, 0, 15, 20, 24, 18, 30, 26, 38, 29],
    [25, 15, 0, 14, 20, 16, 27, 22, 31, 25],
    [18, 20, 14, 0, 16, 19, 23, 21, 29, 24],
    [30, 24, 20, 16, 0, 13, 19, 17, 25, 20],
    [22, 18, 16, 19, 13, 0, 17, 15, 23, 18],
    [35, 30, 27, 23, 19, 17, 0, 12, 20, 16],
    [28, 26, 22, 21, 17, 15, 12, 0, 18, 14],
    [40, 38, 31, 29, 25, 23, 20, 18, 0, 11],
    [32, 29, 25, 24, 20, 18, 16, 14, 11, 0]
], dtype=float)

# Peso de criticidade dos pares: pares próximos à diagonal recebem peso 1,
# e alguns pares críticos recebem peso maior.
P = np.ones((N, N))
for i in range(N):
    for j in range(N):
        if i != j and abs(i-j) <= 2:
            P[i, j] = 2.0

for i, j in [(0, 9), (1, 8), (2, 7), (3, 6)]:
    P[i, j] = P[j, i] = 3.0

print('Matriz D (latências em ms):')
display(pd.DataFrame(D).round(1))


## Função objetivo e validação de árvore

A topologia é avaliada pelo custo ponderado das arestas. A matriz `P` representa a criticidade dos pares: arestas entre switches mais críticos têm maior impacto no objetivo.

Durante a construção, uma aresta só é adicionada quando não cria ciclo. Ao final, uma solução válida deve ter 9 arestas e ser conectada.

In [ ]:
def encontrar_raiz(parent, x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x

def unir(parent, a, b):
    ra = encontrar_raiz(parent, a)
    rb = encontrar_raiz(parent, b)
    if ra == rb:
        return False
    parent[rb] = ra
    return True

def custo_aresta(i, j):
    return D[i, j] * P[i, j]

def custo_topologia(arestas):
    return sum(custo_aresta(i, j) for i, j in arestas)

def arvore_valida(arestas):
    if len(arestas) != N - 1:
        return False
    parent = list(range(N))
    for i, j in arestas:
        if not unir(parent, i, j):
            return False
    raiz = encontrar_raiz(parent, 0)
    return all(encontrar_raiz(parent, i) == raiz for i in range(N))

def construir_arvore(tau):
    arestas = []
    parent = list(range(N))
    candidatos = [(i, j) for i in range(N) for j in range(i+1, N)]

    while len(arestas) < N - 1:
        validas = []
        pesos = []
        for i, j in candidatos:
            ri = encontrar_raiz(parent, i)
            rj = encontrar_raiz(parent, j)
            if ri != rj:
                heuristica = 1.0 / (D[i, j] * P[i, j] + 1e-9)
                peso = (tau[i, j] ** ALPHA) * (heuristica ** BETA)
                validas.append((i, j))
                pesos.append(peso)

        pesos = np.array(pesos, dtype=float)
        pesos /= pesos.sum()
        idx = np.random.choice(len(validas), p=pesos)
        aresta = validas[idx]
        i, j = aresta
        unir(parent, i, j)
        arestas.append(aresta)
        candidatos.remove(aresta)

    return arestas


## Execução do ACO

In [ ]:
tau = np.ones((N, N), dtype=float)
best_global = None
best_global_custo = np.inf
historico = []

for iteracao in range(N_ITER):
    solucoes = []

    for _ in range(N_FORMIGAS):
        arvore = construir_arvore(tau)
        custo = custo_topologia(arvore)
        solucoes.append((custo, arvore))

        if custo < best_global_custo:
            best_global_custo = custo
            best_global = arvore.copy()

    # Evaporação do feromônio.
    tau *= (1 - RHO)

    # Atualização apenas pelas melhores topologias da iteração (elite).
    solucoes.sort(key=lambda x: x[0])
    elite = solucoes[:max(1, N_FORMIGAS // 5)]
    for custo, arvore in elite:
        deposito = Q / (custo + 1e-9)
        for i, j in arvore:
            tau[i, j] += deposito
            tau[j, i] = tau[i, j]

    historico.append(best_global_custo)

print('Melhor custo encontrado:', round(best_global_custo, 4))
print('Árvore válida:', arvore_valida(best_global))
print('Arestas:', [(i+1, j+1) for i, j in best_global])


## Matriz de adjacência final

In [ ]:
adj = np.zeros((N, N), dtype=int)
for i, j in best_global:
    adj[i, j] = 1
    adj[j, i] = 1

print('Matriz de Adjacência final:')
display(pd.DataFrame(adj, index=[f'S{i+1}' for i in range(N)], columns=[f'S{i+1}' for i in range(N)]))

print('Número de arestas:', int(adj.sum() / 2))
print('Conectada e sem ciclos:', arvore_valida(best_global))


## Convergência do ACO

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(range(1, N_ITER + 1), historico)
plt.title('Convergência do ACO')
plt.xlabel('Iteração')
plt.ylabel('Melhor custo acumulado')
plt.grid(True, alpha=0.3)
plt.show()


## Comparação com uma topologia aleatória

A topologia aleatória também será construída como uma árvore geradora válida. Para reduzir o efeito de uma única amostra, são geradas várias árvores aleatórias e usada a média dos custos como referência.

In [ ]:
def arvore_aleatoria():
    arestas = []
    parent = list(range(N))
    candidatos = [(i, j) for i in range(N) for j in range(i+1, N)]
    np.random.shuffle(candidatos)

    for i, j in candidatos:
        if unir(parent, i, j):
            arestas.append((i, j))
            if len(arestas) == N - 1:
                break
    return arestas

np.random.seed(123)
N_ALEATORIAS = 100
custos_aleatorios = [custo_topologia(arvore_aleatoria()) for _ in range(N_ALEATORIAS)]
custo_aleatorio_medio = np.mean(custos_aleatorios)
ganho_percentual = ((custo_aleatorio_medio - best_global_custo) / custo_aleatorio_medio) * 100

comparacao = pd.DataFrame({
    'Topologia': ['ACO', 'Aleatória (média de 100)'],
    'Custo de latência ponderado': [best_global_custo, custo_aleatorio_medio]
})

display(comparacao.round(4))
print(f'Ganho percentual de redução de latência: {ganho_percentual:.2f}%')


## Análise técnica

- O ACO constrói soluções incrementalmente, escolhendo arestas de acordo com feromônio e informação heurística de latência.
- A verificação com Union-Find impede que uma nova aresta forme ciclo durante a construção.
- Uma árvore geradora com 10 switches possui exatamente 9 arestas; portanto, a solução final é validada por quantidade de arestas, ausência de ciclos e conectividade.
- O feromônio sofre evaporação com `rho = 0.2`.
- A atualização de feromônio é feita somente pelas melhores topologias de cada iteração, conforme solicitado.
- O ganho percentual compara o custo da melhor solução do ACO com o custo médio de 100 árvores geradas aleatoriamente.